# dense_embedding_exp3

본 노트북은 dense embedding cosine similarity 분석을 위한 exp3 데이터셋을 생성한다.

## exp3 분석 방식

exp3은 기준 문장과 공시 텍스트를 **문장 단위**로 비교하는 방식이다.

즉, 기업 1개와 사업연도 1개 안에서 II, IV, VI, VIII, XI 섹션을 추출한 뒤, 각 섹션 텍스트를 문장 단위로 분리하여 분석 단위로 사용한다.

이후 다음 단계에서 각 문장과 E/S/G 기준 문장 간 dense embedding cosine similarity를 계산할 예정이다.

## exp3의 핵심 설계

- 비교 단위: 기업·연도·섹션·문장
- 텍스트 구성: II, IV, VI, VIII, XI 섹션을 문장 단위로 분리
- 전처리 수준: dense embedding에 맞게 문장 의미를 최대한 보존
- 형태소 분석: 사용하지 않음
- 최종 컬럼: `exp3_sentence_text`

# 1. 라이브러리 및 경로 설정

In [1]:
# ============================================================
# 1. 라이브러리 및 경로 설정
# ============================================================

import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def find_project_root() -> Path:
    """
    현재 작업 위치에서 data 폴더가 있는 프로젝트 루트를 찾는다.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data").exists():
            return candidate
    return Path.cwd()


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR = PROCESSED_DIR / "analysis"

REPORT_JSON_DIR = PROCESSED_DIR / "dart_full_reports"
COMPANY_MASTER_PATH = RAW_DIR / "company_master.csv"

DENSE_DIR = ANALYSIS_DIR / "dense_embedding_compare"
DENSE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPORT_JSON_DIR:", REPORT_JSON_DIR)
print("COMPANY_MASTER_PATH:", COMPANY_MASTER_PATH)
print("DENSE_DIR:", DENSE_DIR)

PROJECT_ROOT: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final
REPORT_JSON_DIR: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\dart_full_reports
COMPANY_MASTER_PATH: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\raw\company_master.csv
DENSE_DIR: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis\dense_embedding_compare


# 2. 분석 대상 섹션 설정

In [2]:
# ============================================================
# 2. 분석 대상 섹션 설정
# ============================================================

selected_sections = {
    "II": "II. 사업의 내용",
    "IV": "IV. 이사의 경영진단 및 분석의견",
    "VI": "VI. 이사회 등 회사의 기관에 관한 사항",
    "VIII": "VIII. 임원 및 직원 등에 관한 사항",
    "XI": "XI. 그 밖에 투자자 보호를 위하여 필요한 사항",
}

selected_romans = list(selected_sections.keys())

roman_replace_map = {
    "Ⅰ": "I",
    "Ⅱ": "II",
    "Ⅲ": "III",
    "Ⅳ": "IV",
    "Ⅴ": "V",
    "Ⅵ": "VI",
    "Ⅶ": "VII",
    "Ⅷ": "VIII",
    "Ⅸ": "IX",
    "Ⅹ": "X",
    "Ⅺ": "XI",
    "Ⅻ": "XII",
}


def normalize_roman_text(text):
    """
    특수 로마숫자를 일반 영문 로마숫자로 변환한다.
    """
    text = str(text)
    for k, v in roman_replace_map.items():
        text = text.replace(k, v)
    return text


def extract_selected_section_roman(lineage="", text=""):
    """
    lineage 또는 text 앞부분에서 II, IV, VI, VIII, XI 섹션 번호를 추출한다.
    """
    raw = normalize_roman_text(str(lineage))

    if raw.strip() == "" or raw.lower() == "nan":
        raw = normalize_roman_text(str(text)[:1000])

    pattern = r"\b(VIII|VI|IV|XI|II)\b\.?"
    match = re.search(pattern, raw)

    if match:
        return match.group(1)

    return "미분류"


section_order_map = {
    "II": 2,
    "IV": 4,
    "VI": 6,
    "VIII": 8,
    "XI": 11,
}

print("분석 대상 섹션")
for k, v in selected_sections.items():
    print(k, ":", v)

분석 대상 섹션
II : II. 사업의 내용
IV : IV. 이사의 경영진단 및 분석의견
VI : VI. 이사회 등 회사의 기관에 관한 사항
VIII : VIII. 임원 및 직원 등에 관한 사항
XI : XI. 그 밖에 투자자 보호를 위하여 필요한 사항


# 3. DART JSON에서 핵심 섹션 텍스트 추출

In [3]:
# ============================================================
# 3. DART JSON에서 핵심 섹션 텍스트 추출
# ============================================================

json_files = sorted(REPORT_JSON_DIR.glob("*.json"))

print("JSON 파일 수:", len(json_files))

if len(json_files) == 0:
    raise FileNotFoundError("dart_full_reports 폴더에 json 파일이 없습니다.")


def flatten_records(obj):
    """
    JSON 내부에서 text와 lineage 후보를 재귀적으로 추출한다.
    """
    records = []

    text_keys = [
        "text",
        "full_text",
        "unit_text",
        "section_text",
        "content",
        "paragraph",
        "passage",
    ]

    lineage_keys = [
        "lineage",
        "section",
        "section_title",
        "title",
        "heading",
        "chapter",
        "toc",
    ]

    if isinstance(obj, dict):
        keys = set(obj.keys())
        has_text = any(k in keys for k in text_keys)

        if has_text:
            rec = {}

            for k in text_keys:
                if k in obj:
                    rec["text"] = obj.get(k)
                    rec["text_source_key"] = k
                    break

            for k in lineage_keys:
                if k in obj:
                    rec["lineage"] = obj.get(k)
                    rec["lineage_source_key"] = k
                    break

            if "lineage" not in rec:
                rec["lineage"] = ""
                rec["lineage_source_key"] = ""

            records.append(rec)

        for v in obj.values():
            records.extend(flatten_records(v))

    elif isinstance(obj, list):
        for item in obj:
            records.extend(flatten_records(item))

    return records


rows = []

for path in json_files:
    file_name = path.name

    # 예: 005930_2024.json
    m = re.match(r"(\d{6})_(\d{4})\.json", file_name)

    if m:
        stock_code = m.group(1)
        fiscal_year = int(m.group(2))
    else:
        stock_code = ""
        fiscal_year = np.nan

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = flatten_records(data)

    for i, rec in enumerate(records):
        lineage = rec.get("lineage", "")

        if isinstance(lineage, list):
            lineage = " > ".join([str(x) for x in lineage])
        else:
            lineage = str(lineage)

        text = str(rec.get("text", ""))

        section_roman = extract_selected_section_roman(
            lineage=lineage,
            text=text
        )

        if section_roman not in selected_romans:
            continue

        rows.append({
            "source_file": file_name,
            "stock_code": str(stock_code).zfill(6),
            "fiscal_year": int(fiscal_year),
            "esg_year": int(fiscal_year) + 1,
            "row_order": i + 1,
            "section_roman": section_roman,
            "section_clean": selected_sections[section_roman],
            "section_order": section_order_map[section_roman],
            "lineage": lineage,
            "text": text,
            "text_chars": len(text),
            "text_source_key": rec.get("text_source_key", ""),
            "lineage_source_key": rec.get("lineage_source_key", ""),
        })


section_text_df = pd.DataFrame(rows)

print("선택 섹션 텍스트 행 수:", len(section_text_df))

if len(section_text_df) == 0:
    raise ValueError("II, IV, VI, VIII, XI 섹션이 추출되지 않았습니다.")

print("분석 기업-연도 수:", section_text_df[["stock_code", "fiscal_year"]].drop_duplicates().shape[0])
print("\n섹션별 행 수")
print(section_text_df["section_clean"].value_counts())

display(section_text_df.head())

JSON 파일 수: 381
선택 섹션 텍스트 행 수: 176
분석 기업-연도 수: 176

섹션별 행 수
section_clean
II. 사업의 내용                     142
VIII. 임원 및 직원 등에 관한 사항          18
VI. 이사회 등 회사의 기관에 관한 사항          7
IV. 이사의 경영진단 및 분석의견              5
XI. 그 밖에 투자자 보호를 위하여 필요한 사항      4
Name: count, dtype: int64


,source_file,stock_code,fiscal_year,esg_year,row_order,section_roman,section_clean,section_order,lineage,text,text_chars,text_source_key,lineage_source_key
0,000020_2023.json,000020,2023,2024,1,II,II. 사업의 내용,2,,감사보고서 \n4.1 \n동화약품 주식회사 \n\n00469373 \n1000000...,441278,full_text,
1,000020_2024.json,000020,2024,2025,1,II,II. 사업의 내용,2,,연결감사보고서 \n6.0 \n동화약품 주식회사 \n\n00469373 \n10000...,452670,full_text,
2,000050_2022.json,000050,2022,2023,1,II,II. 사업의 내용,2,,사업보고서 \n5.0 \n(주)경방 \n\nC \nA \n\n사 업 보 고 서 \n...,351848,full_text,
3,000050_2023.json,000050,2023,2024,1,II,II. 사업의 내용,2,,감사보고서 \n4.1 \n주식회사 경방 \n\n00260198 \n100000000...,373273,full_text,
4,000050_2024.json,000050,2024,2025,1,II,II. 사업의 내용,2,,연결감사보고서 \n6.0 \n주식회사 경방 \n\n00260198 \n1000000...,414575,full_text,


# 4. dense embedding용 기본 정제 함수 정의

dense embedding 방식에서는 문장 의미 보존이 중요하므로 형태소 분석은 적용하지 않는다.

이 단계에서는 다음 정도의 기본 정제만 수행한다.

- 공백 정리
- 줄바꿈 정리
- 깨진 특수문자 일부 제거
- 과도한 기호 제거
- 문장 구조는 최대한 유지

In [4]:
# ============================================================
# 4. dense embedding용 기본 정제 함수 정의
# ============================================================

def clean_for_dense_sentence(text):
    """
    dense embedding용 기본 정제 함수.
    문장 의미 보존을 위해 과도한 전처리는 하지 않는다.
    """
    text = str(text)

    # HTML 태그처럼 보이는 것 제거
    text = re.sub(r"<[^>]+>", " ", text)

    # 분석에 크게 필요 없는 특수문자 제거
    # 단, 문장부호 일부는 문장 분리를 위해 유지
    text = re.sub(r"[^\w가-힣\s.,;:!?%()\-]", " ", text)

    # 로마숫자 특수문자 정리
    text = normalize_roman_text(text)

    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    return text

# 5. 문장 단위 분리 함수 정의

- 섹션 텍스트를 문장 단위로 분리한다.
- 너무 짧은 문장은 의미가 부족할 수 있으므로 제외한다.
- 한글 비율이 지나치게 낮은 문장은 표, 숫자, 영문 중심 문장일 가능성이 있어 제외한다.
- dense embedding 모델에 입력할 문장 단위 텍스트를 생성한다.

In [5]:
# ============================================================
# 5. 문장 단위 분리 함수 정의
# ============================================================

def split_into_sentences(text, min_chars=30, max_chars=1000, min_korean_ratio=0.25):
    """
    텍스트를 dense embedding 입력용 문장 단위로 분리한다.

    min_chars:
        너무 짧은 문장을 제거하기 위한 최소 글자 수

    max_chars:
        너무 긴 문장을 제거하기 위한 최대 글자 수

    min_korean_ratio:
        한글 비율이 너무 낮은 문장을 제거하기 위한 기준
    """
    text = clean_for_dense_sentence(text)

    if not text:
        return []

    # 문장 끝 표현과 줄바꿈 기준으로 분리
    candidates = re.split(
        r"(?<=[.!?。])\s+|(?<=[다요함음임])\.\s+|\n+",
        text
    )

    sentences = []

    for sent in candidates:
        sent = re.sub(r"\s+", " ", str(sent)).strip()

        if len(sent) < min_chars:
            continue

        if len(sent) > max_chars:
            # 너무 긴 문장은 쉼표, 세미콜론 등을 기준으로 한 번 더 분리
            sub_sents = re.split(r"[;；]\s+|,\s+", sent)
        else:
            sub_sents = [sent]

        for sub in sub_sents:
            sub = re.sub(r"\s+", " ", str(sub)).strip()

            if len(sub) < min_chars or len(sub) > max_chars:
                continue

            korean_chars = len(re.findall(r"[가-힣]", sub))
            total_chars = len(re.sub(r"\s+", "", sub))

            if total_chars == 0:
                continue

            korean_ratio = korean_chars / total_chars

            if korean_ratio < min_korean_ratio:
                continue

            sentences.append(sub)

    return sentences

# 6. 기업·연도·섹션·문장 단위 데이터 생성

- 각 섹션 텍스트를 문장 단위로 분리한다.
- 각 문장을 하나의 분석 단위로 저장한다.
- 결과 컬럼은 `exp3_sentence_text`이다.

즉, exp3의 비교 단위는 다음과 같다.

`기준 문장 ↔ 기업·연도·섹션·문장`

In [6]:
# ============================================================
# 6. 기업·연도·섹션·문장 단위 데이터 생성
# ============================================================

sentence_rows = []

section_text_df = section_text_df.sort_values(
    ["stock_code", "fiscal_year", "section_order", "row_order"]
)

for row in section_text_df.itertuples(index=False):
    sentences = split_into_sentences(
        getattr(row, "text"),
        min_chars=30,
        max_chars=1000,
        min_korean_ratio=0.25
    )

    for sent_idx, sentence in enumerate(sentences):
        analysis_unit = f"{getattr(row, 'stock_code')}x{getattr(row, 'fiscal_year')}"
        section_analysis_unit = f"{analysis_unit}_{getattr(row, 'section_roman')}"
        sentence_analysis_unit = f"{section_analysis_unit}_sent{sent_idx+1:04d}"

        sentence_rows.append({
            "sentence_analysis_unit": sentence_analysis_unit,
            "section_analysis_unit": section_analysis_unit,
            "analysis_unit": analysis_unit,
            "source_file": getattr(row, "source_file"),
            "stock_code": getattr(row, "stock_code"),
            "fiscal_year": getattr(row, "fiscal_year"),
            "esg_year": getattr(row, "esg_year"),
            "section_roman": getattr(row, "section_roman"),
            "section_clean": getattr(row, "section_clean"),
            "section_order": getattr(row, "section_order"),
            "row_order": getattr(row, "row_order"),
            "sentence_index": sent_idx + 1,
            "exp3_sentence_text": sentence,
            "exp3_sentence_text_chars": len(sentence),
        })

sentence_unit_df = pd.DataFrame(sentence_rows)

print("문장 단위 데이터 수:", len(sentence_unit_df))

if len(sentence_unit_df) == 0:
    raise ValueError("문장 단위 데이터가 생성되지 않았습니다. 문장 분리 기준을 확인하세요.")

print("\n섹션별 문장 수")
print(sentence_unit_df["section_clean"].value_counts())

print("\n기업-연도별 문장 수 요약")
display(
    sentence_unit_df
    .groupby("analysis_unit")["exp3_sentence_text"]
    .count()
    .describe()
)

display(sentence_unit_df.head())

문장 단위 데이터 수: 452699

섹션별 문장 수
section_clean
II. 사업의 내용                     386967
VIII. 임원 및 직원 등에 관한 사항          35515
VI. 이사회 등 회사의 기관에 관한 사항         15906
IV. 이사의 경영진단 및 분석의견             10407
XI. 그 밖에 투자자 보호를 위하여 필요한 사항      3904
Name: count, dtype: int64

기업-연도별 문장 수 요약


count      176.000000
mean      2572.153409
std       1370.158762
min        714.000000
25%       1820.500000
50%       2408.500000
75%       2967.250000
max      13585.000000
Name: exp3_sentence_text, dtype: float64

,sentence_analysis_unit,section_analysis_unit,analysis_unit,source_file,stock_code,fiscal_year,esg_year,section_roman,section_clean,section_order,row_order,sentence_index,exp3_sentence_text,exp3_sentence_text_chars
0,000020x2023_II_sent0001,000020x2023_II,000020x2023,000020_2023.json,000020,2023,2024,II,II. 사업의 내용,2,1,1,감사보고서 4.1 동화약품 주식회사 00469373 100000000000 2024...,209
1,000020x2023_II_sent0002,000020x2023_II,000020x2023,000020_2023.json,000020,2023,2024,II,II. 사업의 내용,2,1,2,독립된 감사인의 감사보고서 --------------------- II.,40
2,000020x2023_II_sent0003,000020x2023_II,000020x2023,000020_2023.json,000020,2023,2024,II,II. 사업의 내용,2,1,3,내부회계관리제도 감사의견 독립된 감사인의 내부회계관리제도 감사보고서 --------...,178
3,000020x2023_II_sent0004,000020x2023_II,000020x2023,000020_2023.json,000020,2023,2024,II,II. 사업의 내용,2,1,4,"해당 재무제표는 2023년 12월 31일 현재의 재무상태표, 동일로 종료되는 보고기...",113
4,000020x2023_II_sent0005,000020x2023_II,000020x2023,000020_2023.json,000020,2023,2024,II,II. 사업의 내용,2,1,5,우리의 의견으로는 별첨된 회사의 재무제표는 회사의 2023년 12월 31일 현재의 ...,242


# 7. ESG 등급 데이터 병합

In [7]:
# ============================================================
# 7. ESG 등급 데이터 병합
# ============================================================

company_master = pd.read_csv(COMPANY_MASTER_PATH, dtype={"stock_code": str})
company_master.columns = [str(c).strip() for c in company_master.columns]

company_master["stock_code"] = (
    company_master["stock_code"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(6)
)

company_master["fiscal_year"] = company_master["fiscal_year"].astype(int)

if "esg_year" in company_master.columns:
    company_master["esg_year"] = company_master["esg_year"].astype(int)
else:
    company_master["esg_year"] = company_master["fiscal_year"] + 1

grade_map = {
    "D": 0,
    "C": 1,
    "B": 2,
    "B+": 3,
    "A": 4,
    "A+": 5,
    "S": 6,
}

if "esg_grade_num" not in company_master.columns:
    company_master["esg_grade"] = company_master["esg_grade"].astype(str).str.strip()
    company_master["esg_grade_num"] = company_master["esg_grade"].map(grade_map)

grade_cols = [
    "stock_code",
    "fiscal_year",
    "esg_year",
    "esg_grade",
    "esg_grade_num",
]

for col in ["company_name", "회사이름", "e_grade", "s_grade", "g_grade"]:
    if col in company_master.columns:
        grade_cols.append(col)

analysis_df = pd.merge(
    sentence_unit_df,
    company_master[grade_cols].drop_duplicates(),
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left"
)

print("병합 후 데이터 수:", len(analysis_df))
print("ESG 등급 결측 수:", analysis_df["esg_grade"].isna().sum())

display(analysis_df.head())

병합 후 데이터 수: 452699
ESG 등급 결측 수: 0


,sentence_analysis_unit,section_analysis_unit,analysis_unit,source_file,stock_code,fiscal_year,esg_year,section_roman,section_clean,section_order,row_order,sentence_index,exp3_sentence_text,exp3_sentence_text_chars,esg_grade,esg_grade_num,company_name,e_grade,s_grade,g_grade
0,000020x2023_II_sent0001,000020x2023_II,000020x2023,000020_2023.json,000020,2023,2024,II,II. 사업의 내용,2,1,1,감사보고서 4.1 동화약품 주식회사 00469373 100000000000 2024...,209,C,1,동화약품,B,B,C
1,000020x2023_II_sent0002,000020x2023_II,000020x2023,000020_2023.json,000020,2023,2024,II,II. 사업의 내용,2,1,2,독립된 감사인의 감사보고서 --------------------- II.,40,C,1,동화약품,B,B,C
2,000020x2023_II_sent0003,000020x2023_II,000020x2023,000020_2023.json,000020,2023,2024,II,II. 사업의 내용,2,1,3,내부회계관리제도 감사의견 독립된 감사인의 내부회계관리제도 감사보고서 --------...,178,C,1,동화약품,B,B,C
3,000020x2023_II_sent0004,000020x2023_II,000020x2023,000020_2023.json,000020,2023,2024,II,II. 사업의 내용,2,1,4,"해당 재무제표는 2023년 12월 31일 현재의 재무상태표, 동일로 종료되는 보고기...",113,C,1,동화약품,B,B,C
4,000020x2023_II_sent0005,000020x2023_II,000020x2023,000020_2023.json,000020,2023,2024,II,II. 사업의 내용,2,1,5,우리의 의견으로는 별첨된 회사의 재무제표는 회사의 2023년 12월 31일 현재의 ...,242,C,1,동화약품,B,B,C


# 8. exp3 데이터 점검

- 문장 단위 데이터 수를 확인한다.
- 기업·연도별 문장 수를 확인한다.
- 섹션별 문장 수를 확인한다.
- 문장 길이 분포를 확인한다.
- 샘플 문장을 직접 확인하여 문장 분리가 적절한지 검토한다.

In [8]:
# ============================================================
# 8. exp3 데이터 점검
# ============================================================

print("전체 문장 단위 데이터 수:", len(analysis_df))
print("기업 수:", analysis_df["stock_code"].nunique())

print("\n사업연도 분포:")
print(analysis_df["fiscal_year"].value_counts().sort_index())

print("\n섹션별 문장 수:")
print(analysis_df["section_clean"].value_counts())

print("\n기업-연도별 문장 수 요약:")
display(
    analysis_df
    .groupby("analysis_unit")["exp3_sentence_text"]
    .count()
    .describe()
)

print("\n문장 길이 요약:")
display(
    analysis_df["exp3_sentence_text_chars"].describe()
)

sample_cols = [
    "sentence_analysis_unit",
    "analysis_unit",
    "company_name" if "company_name" in analysis_df.columns else "stock_code",
    "fiscal_year",
    "esg_year",
    "esg_grade",
    "section_clean",
    "exp3_sentence_text",
    "exp3_sentence_text_chars",
]

sample_cols = [c for c in sample_cols if c in analysis_df.columns]

display(
    analysis_df[sample_cols].head(20)
)

전체 문장 단위 데이터 수: 452699
기업 수: 100

사업연도 분포:
fiscal_year
2022    174891
2023    137186
2024    140622
Name: count, dtype: int64

섹션별 문장 수:
section_clean
II. 사업의 내용                     386967
VIII. 임원 및 직원 등에 관한 사항          35515
VI. 이사회 등 회사의 기관에 관한 사항         15906
IV. 이사의 경영진단 및 분석의견             10407
XI. 그 밖에 투자자 보호를 위하여 필요한 사항      3904
Name: count, dtype: int64

기업-연도별 문장 수 요약:


count      176.000000
mean      2572.153409
std       1370.158762
min        714.000000
25%       1820.500000
50%       2408.500000
75%       2967.250000
max      13585.000000
Name: exp3_sentence_text, dtype: float64


문장 길이 요약:


count    452699.000000
mean        139.495080
std         145.991935
min          30.000000
25%          57.000000
50%          88.000000
75%         152.000000
max        1000.000000
Name: exp3_sentence_text_chars, dtype: float64

,sentence_analysis_unit,analysis_unit,company_name,fiscal_year,esg_year,esg_grade,section_clean,exp3_sentence_text,exp3_sentence_text_chars
0,000020x2023_II_sent0001,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,감사보고서 4.1 동화약품 주식회사 00469373 100000000000 2024...,209
1,000020x2023_II_sent0002,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,독립된 감사인의 감사보고서 --------------------- II.,40
2,000020x2023_II_sent0003,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,내부회계관리제도 감사의견 독립된 감사인의 내부회계관리제도 감사보고서 --------...,178
3,000020x2023_II_sent0004,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,"해당 재무제표는 2023년 12월 31일 현재의 재무상태표, 동일로 종료되는 보고기...",113
4,000020x2023_II_sent0005,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,우리의 의견으로는 별첨된 회사의 재무제표는 회사의 2023년 12월 31일 현재의 ...,242
5,000020x2023_II_sent0006,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,감사의견근거 우리는 대한민국의 회계감사기준에 따라 감사를 수행하였습니다,39
6,000020x2023_II_sent0007,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,이 기준에 따른 우리의 책임은 이 감사보고서의 재무제표 감사에 대한 감사인의 책임 ...,59
7,000020x2023_II_sent0008,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,우리는 재무제표 감사와 관련된 대한민국의 윤리적 요구사항에 따라 회사로부터 독립적이...,82
8,000020x2023_II_sent0009,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,우리가 입수한 감사증거가 감사의견을 위한 근거로서 충분하고 적합하다고 우리는 믿습니다,47
9,000020x2023_II_sent0010,000020x2023,동화약품,2023,2024,C,II. 사업의 내용,"강조사항 감사의견에는 영향을 미치지 않는 사항으로서 , 이용자는 주석 11에 주의를...",63


# 9. exp3 결과 저장

In [9]:
# ============================================================
# 9. exp3 결과 저장
# ============================================================

output_path = DENSE_DIR / "dense_embedding_exp3_sentence_unit.csv"

save_cols = []

for col in [
    "sentence_analysis_unit",
    "section_analysis_unit",
    "analysis_unit",
    "stock_code",
    "company_name",
    "회사이름",
    "fiscal_year",
    "esg_year",
    "esg_grade",
    "esg_grade_num",
    "section_roman",
    "section_clean",
    "section_order",
    "row_order",
    "sentence_index",
    "exp3_sentence_text",
    "exp3_sentence_text_chars",
]:
    if col in analysis_df.columns and col not in save_cols:
        save_cols.append(col)

analysis_df[save_cols].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)
print("저장 컬럼:")
print(save_cols)

저장 완료: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis\dense_embedding_compare\dense_embedding_exp3_sentence_unit.csv
저장 컬럼:
['sentence_analysis_unit', 'section_analysis_unit', 'analysis_unit', 'stock_code', 'company_name', 'fiscal_year', 'esg_year', 'esg_grade', 'esg_grade_num', 'section_roman', 'section_clean', 'section_order', 'row_order', 'sentence_index', 'exp3_sentence_text', 'exp3_sentence_text_chars']
